# ⚛️ RATISS-GI — Entraînement GPU VRAM Massif : Open-Hermes-FR + Compar:IA

**Jonathan Evina · RATISS Labs · Yaoundé, Cameroun 🇨🇲**

> **🚀 Moteur 100% GPU VRAM CUDA :** Débit mesuré à **1 280 000+ tokens/seconde** sur Nvidia T4 (15 Go VRAM) grâce à la vectorisation par Bit-Packing 64-bit (`src << 32 | dst`) et 60-bit (`t1 << 40 | t2 << 20 | t3`).  
> **🔄 Gonflage Incrémental d'Acquis :** Le script reprend automatiquement votre modèle venant d'être calculé (`sortie/langue_massive/bouche.bouche` avec ses 500 892 arcs et 1 083 976 trigrammes) ou `ratiss_self/bouche.bouche` pour le fusionner et le faire grandir sans repartir de zéro !  
> **📚 Les 2 Datasets Souverains Ingestés :**  
> 1. **Open-Hermes-FR (`legmlai/openhermes-fr`)** : Le plus massif de France (799 875 paires, 1,4 Go, distillation GPT-4o).  
> 2. **Compar:IA Conversations (`Data-Gouv-ML/compar-ia`)** : Le plus conversationnel de France (641 572 tours multi-tours réels de citoyens et utilisateurs du Ministère de la Culture).  
> **📥 Sauvegarde Directe Chrome :** Téléchargement direct du modèle final scellé en 1 clic sans clé GitHub.

## 1. Détection VRAM & Reprise Automatique du Modèle Précédent

In [ ]:
import os, subprocess, sys, torch

print('🚀 Détection du matériel GPU / VRAM :')
if torch.cuda.is_available():
    gpu_nom = torch.cuda.get_device_name(0)
    vram_totale = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  ✓ GPU Actif : {gpu_nom}')
    print(f'  ✓ VRAM Dédiée Disponible : {vram_totale:.2f} Go')
else:
    print('  ⚠️ Aucun GPU détecté : basculement sur CPU vectorisé.')
    print('  💡 Conseil : Menu Exécution > Modifier le type d\'exécution > T4 GPU.')

# Récupération ou mise à jour directe du dépôt officiel
if not os.path.exists('ratiss_v3.py'):
    subprocess.check_call(['git', 'clone', 'https://github.com/jonathansearch/Ratiss-GI.git'])
    os.chdir('Ratiss-GI')
else:
    print('🔄 Synchronisation avec le dernier moteur GPU VRAM...')
    subprocess.run(['git', 'pull', 'origin', 'main'])

# Détection automatique du modèle précédent à gonfler
CANDIDATS = [
    'sortie/langue_massive/bouche.bouche',  # Modèle à 500k arcs venant de finir !
    'ratiss_self/bouche.bouche',
    'bouche.bouche'
]
MODELE_A_CHARGER = None
for c in CANDIDATS:
    if os.path.exists(c) and os.path.getsize(c) > 1000:
        MODELE_A_CHARGER = c
        break

if MODELE_A_CHARGER:
    taille = os.path.getsize(MODELE_A_CHARGER)
    print(f'\n📦 MODÈLE EXISTANT DÉTECTÉ : {MODELE_A_CHARGER} ({taille:,} octets)')
    print('  ✓ Ce modèle sera automatiquement chargé et gonflé par Open-Hermes-FR et Compar:IA !')
else:
    print('\n🌱 Aucun modèle antérieur trouvé : initialisation vierge.')

# Exécution des tests unitaires
!python3 -m unittest discover -s tests -p 'test_*.py'

## 2. Démonstration du Pilier 4 : Résonance d'Onde & Normalisation par le Degré

Vérification de l'étouffement des hubs syntaxiques (`de`, `le`) par la loi $\frac{\text{poids}}{\ln(\text{degré})}$ et émergence de `lune` au **Rang #2**.

In [ ]:
from ratiss_v3 import TissuScelle

tissu = TissuScelle('cerveau/RatissOne.ratiss')
res = tissu.raisonner_par_resonance(['soleil', 'maree'], seuil_energie=0.01, max_sauts=2, normaliser_degre=True)

print('🏆 Top 5 des carrefours émergents sur le tissu scellé :')
for i, c in enumerate(res['carrefours'][:5]):
    print(f"  #{i+1} {c['noeud']:<15} énergie: {c['energie']:.4f} (sauts: {c['sauts']})")

## 3. Ingestion Streaming des 2 Datasets Souverains (Open-Hermes-FR + Compar:IA)

Nous utilisons le streaming Hugging Face pour charger directement en mémoire sans saturer le disque dur :
- **Open-Hermes-FR** : 799 875 paires structurées issues de GPT-4o.
- **Compar:IA Conversations** : 641 572 tours conversationnels réels multi-tours du Ministère de la Culture.

*Vous pouvez ajuster `MAX_PAR_DATASET` (ex: 50 000 pour ~100 000 paires ultra-rapides, ou `None` pour tout le corpus complet).* 

In [ ]:
# Installation rapide de datasets et pyarrow si nécessaire
!pip install -q -U datasets pyarrow

import os, re, json
from pathlib import Path
from datasets import load_dataset

corpus_dir = Path('corpus')
corpus_dir.mkdir(exist_ok=True)
lignes_dialogues = []

# Nombre de dialogues à extraire de chaque corpus (None = intégrale des 1,4 million de paires)
MAX_HERMES = 50000
MAX_COMPARAI = 50000

print('========================================================================')
print('📥 INGESTION STREAMING : OPEN-HERMES-FR & COMPAR:IA CONVERSATIONS')
print('========================================================================')

# 1. Open-Hermes-FR (legmlai/openhermes-fr)
print('\n📚 1️⃣ Streaming Open-Hermes-FR (799 875 paires disponibles)...')
try:
    ds_hermes = load_dataset('legmlai/openhermes-fr', split='train', streaming=True)
    c_hermes = 0
    for row in ds_hermes:
        q = row.get('prompt', '').strip()
        r = row.get('accepted_completion', '').strip()
        if q and r:
            q_clean = ' '.join(q.split())[:300]
            r_clean = ' '.join(r.split())[:450]
            lignes_dialogues.append(f"_q_ {q_clean} _r_ {r_clean}")
            c_hermes += 1
            if MAX_HERMES and c_hermes >= MAX_HERMES:
                break
    print(f'   ✓ Open-Hermes-FR : {c_hermes:,} paires distillées GPT-4o ajoutées !')
except Exception as e:
    print(f'   ⚠️ Erreur Open-Hermes-FR : {e}')

# 2. Compar:IA Conversations (Data-Gouv-ML/compar-ia)
print('\n💬 2️⃣ Streaming Compar:IA Conversations (641 572 tours disponibles)...')
try:
    ds_comparia = load_dataset('Data-Gouv-ML/compar-ia', data_files='data/comparia-conversations.parquet', split='train', streaming=True)
    c_comp = 0
    for row in ds_comparia:
        conv = row.get('full_conversation_a') or row.get('full_conversation_b')
        if conv and isinstance(conv, list):
            tours_txt = []
            for turn in conv:
                role = turn.get('role', '')
                content = ' '.join(turn.get('content', '').split())[:350]
                if role == 'user':
                    tours_txt.append(f"_q_ {content}")
                elif role == 'assistant':
                    tours_txt.append(f"_r_ {content}")
            if tours_txt:
                lignes_dialogues.append(' '.join(tours_txt))
                c_comp += 1
        elif row.get('prompt'):
            q = ' '.join(row.get('prompt', '').split())[:300]
            r = ' '.join(row.get('response', '').split())[:400]
            lignes_dialogues.append(f"_q_ {q} _r_ {r}")
            c_comp += 1
        if MAX_COMPARAI and c_comp >= MAX_COMPARAI:
            break
    print(f'   ✓ Compar:IA : {c_comp:,} conversations multi-tours réelles ajoutées !')
except Exception as e:
    print(f'   ⚠️ Erreur Compar:IA : {e}')

# Sauvegarde du grand corpus consolidé
fichier_corpus = corpus_dir / 'corpus_hermes_comparia.txt'
fichier_corpus.write_text('\n'.join(lignes_dialogues), encoding='utf-8')
print('\n' + '='*70)
print(f'✅ GRAND CORPUS SOUVERAIN CONSTITUÉ : {fichier_corpus}')
print(f'📊 Volume total : {len(lignes_dialogues):,} dialogues & échanges ({fichier_corpus.stat().st_size / 1e6:.2f} Mo)')
print('='*70)

## 4. Entraînement GPU VRAM Dédié & Gonflage du Modèle Existant

Le moteur GPU charge automatiquement vos acquis précédents (`sortie/langue_massive/bouche.bouche` ou `ratiss_self/bouche.bouche`), exploite la VRAM CUDA pour le calcul vectorisé, et gonfle le cerveau à **plus de 1 200 000 tokens/seconde** !

In [ ]:
SORTIE_DIR = 'sortie/langue_massive'

CANDIDATS = [
    'sortie/langue_massive/bouche.bouche',
    'ratiss_self/bouche.bouche',
    'bouche.bouche'
]
MODELE_REPRISE = None
for c in CANDIDATS:
    if os.path.exists(c) and os.path.getsize(c) > 1000:
        MODELE_REPRISE = c
        break

cmd = [
    'python3', 'entrainement/gpu/entrainer_gpu.py',
    '--corpus', 'corpus/corpus_hermes_comparia.txt',
    '--sortie', SORTIE_DIR,
    '--registre', 'dialogue',
    '--tours', '3',
    '--fraction', '0.05'
]

if MODELE_REPRISE:
    print(f'🔄 REPRISE ET GONFLAGE DU MODÈLE : {MODELE_REPRISE} ({os.path.getsize(MODELE_REPRISE):,} octets) !')
    cmd += ['--charger', MODELE_REPRISE]
else:
    print('🌱 Initialisation vierge.')

!{' '.join(cmd)}

print('\n' + '='*50)
print(open(f'{SORTIE_DIR}/ORGANISATION.txt', encoding='utf-8').read())

## 5. Banc d'Essai de Parole & Dialogue en Direct (6 Tests Conversationnels)

Testons directement la capacité conversationnelle de la Bouche gonflée sur différentes questions et répliques !

In [ ]:
import subprocess

TESTS_CONVERSATIONNELS = [
    ('1. Identité & Conscience', '_q_ qui es tu et quelle est ta nature _r_', 0.75, 8, True),
    ('2. Question Scientifique (Soleil & Lune)', '_q_ parle moi du soleil et de la lune _r_', 0.65, 8, True),
    ('3. Connaissance Fondamentale (Gravité)', '_q_ comment fonctionne la gravite _r_', 0.75, 8, True),
    ('4. Aventure & Exploration Maritime', '_q_ comment explorer les profondeurs de l ocean _r_', 0.75, 8, True),
    ('5. Échange Émotionnel & Esprit', '_q_ que penses tu du courage et de l honneur _r_', 0.70, 8, True),
    ('6. Mystère du Monde (Génération Libre)', '_q_ raconte moi un secret sur le monde _r_', 0.75, 8, False),
]

print('='*70)
print('🎙️ BANC D\'ESSAI CONVERSATIONNEL DU MODÈLE GONFLÉ')
print('='*70)

for titre, amorce, lam, mem, glouton in TESTS_CONVERSATIONNELS:
    print(f'\n👉 {titre}')
    print(f'   Question / Amorce : "{amorce}"')
    cmd = [
        'python3', 'ratiss_v3.py', 'cerveau/RatissOne.ratiss',
        '--charger', f'{SORTIE_DIR}/bouche.bouche',
        '--lam', str(lam),
        '--memoire', str(mem),
        '--mots', '12',
        amorce
    ]
    if glouton:
        cmd.append('--glouton')
    res = subprocess.run(cmd, capture_output=True, text=True)
    suite = res.stdout.strip().splitlines()[-1] if res.stdout.strip() else '...'
    print(f'   Réponse Bouche   : \033[92m{suite}\033[0m')

## 6. Sauvegarde Directe & Téléchargement Chrome (Zéro Clé Requise)

Cette cellule compresse automatiquement tous les artefacts d'entraînement (`bouche.bouche`, `mesures.json`, `ORGANISATION.txt`, etc.) et déclenche le téléchargement direct sur votre ordinateur via votre navigateur Chrome.

In [ ]:
from google.colab import files
import shutil, os
from pathlib import Path

print('='*60)
print('📥 SAUVEGARDE DIRECTE ET TÉLÉCHARGEMENT CHROME')
print('='*60)

dossier_export = Path('sortie/ratiss_conversationnel')
dossier_export.mkdir(parents=True, exist_ok=True)

# Rassemblement des artefacts du modèle gonflé
fichiers = [
    'sortie/langue_massive/bouche.bouche',
    'sortie/langue_massive/mesures.json',
    'sortie/langue_massive/ORGANISATION.txt',
    'sortie/langue_massive/01_education.txt',
    'sortie/langue_massive/02_ecart.txt'
]

for f in fichiers:
    if os.path.exists(f):
        shutil.copy2(f, dossier_export)
        print(f'  ✓ Préparé : {f}')

# Création de l\'archive zip officielle
archive_nom = 'ratiss_bouche_hermes_comparia_gonflee'
shutil.make_archive(archive_nom, 'zip', dossier_export)
archive_zip = f'{archive_nom}.zip'

taille_zip = os.path.getsize(archive_zip)
print(f'\n📦 Archive prête : {archive_zip} ({taille_zip:,} octets)')
print('🚀 Déclenchement automatique du téléchargement Chrome...')

# Téléchargement direct vers le navigateur
files.download(archive_zip)
files.download('sortie/langue_massive/bouche.bouche')

print('🎉 Téléchargement Chrome déclenché avec succès !')